# CineData — agente de filmes (Text-to-SQL)
Agente em Python + PydanticAI + OpenRouter `:free` que responde em português consultando a Gold (`cinerocket.db`) em tempo real. Este notebook explica cada módulo do `src/cinedata/` na ordem de construção. O código final mora em `src/cinedata/`; aqui mostramos cada parte explicada.
Ordem: exploração do banco → SQLs de referência → config → banco → guardrails → modelos → prompt → cache → agente → cli → demo. Detalhes em `docs/anotacoes_notebook.md`.

## 1. Exploração da Gold
10 tabelas do modelo dimensional + `alembic_version` (controle, fora do escopo). Código lista colunas e 3 linhas por tabela.

In [1]:
import sqlite3
from pathlib import Path

db_path = (Path.cwd() / "../src/cinedata/data/cinerocket.db").resolve()

if not db_path.exists():
    raise FileNotFoundError(f"Banco não encontrado em: {db_path}")

conn = sqlite3.connect(str(db_path))

tabelas = [r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
)]

for t in tabelas:
    print(f"\n== {t} ==")
    for cid, nome, tipo, notnull, default, pk in conn.execute(f"PRAGMA table_info({t})"):
        print(f"  {nome} ({tipo}){' [PK]' if pk else ''}")
    total = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  -> {total} linhas")
    cur = conn.execute(f"SELECT * FROM {t} LIMIT 3")
    print("  colunas:", [d[0] for d in cur.description])
    for linha in cur.fetchall():
        print("   ", linha)


== alembic_version ==
  version_num (VARCHAR(32)) [PK]
  -> 1 linhas
  colunas: ['version_num']
    ('0001_initial_schema',)

== bridge_movie_company ==
  sk_movie_id (VARCHAR(64)) [PK]
  sk_company_id (VARCHAR(64)) [PK]
  -> 116326 linhas
  colunas: ['sk_movie_id', 'sk_company_id']
    ('8a85beff7b0f64692b227466b2c9c4abb274cf1225da4d5ce495da2180ba89c8', 'a93d6d918018600b5fba9d39dbb7898430ece8d5fb29e42a2f8db59a37f5faf7')
    ('c1e53d82aeac91f16040913257678ba690813daa7a5c5d3e182866a339553ce0', 'f0ed91f80406391e38f2e7cc647c452f8f5919e770d7d1247969c27f14bc0980')
    ('ceeda10da2fb7789a0c63741e562f050cec19f48ab727861678a10e51f9fc1f9', 'e8170686c11431164d3ad11ef91043bfb5a67a6a4af9f7e9f6e3ea51476398bd')

== bridge_movie_genre ==
  sk_movie_id (VARCHAR(64)) [PK]
  sk_genre_id (VARCHAR(64)) [PK]
  -> 121521 linhas
  colunas: ['sk_movie_id', 'sk_genre_id']
    ('b5c312c4d8cee94972412b9ea785976f9dbb2c26dd11480d9f2b9b4691df1571', '271a22f75d73e13fa558d991442ecde8d75a34a0b9f6216d44b116f8612c82dc'

## 2. Checagens dos dados
Tipos, anos (2016–2029), NULLs (receita/orçamento), notas e contagens. Base: 95.645 linhas, ~3.373 com receita, ~1.630 com receita+orçamento.

In [2]:
def consultar(sql):
    cur = conn.execute(sql)
    print([d[0] for d in cur.description])
    for linha in cur.fetchall():
        print("  ", linha)
    print()

checagens = [
    "SELECT tipo_pessoa, COUNT(*) FROM dim_people GROUP BY tipo_pessoa",
    "SELECT status_filme, COUNT(*) FROM dim_movies GROUP BY status_filme",
    "SELECT MIN(ano_lancamento), MAX(ano_lancamento) FROM dim_movies",
    """SELECT SUM(receita_usd IS NULL), SUM(receita_usd = 0),
              SUM(orcamento_usd IS NULL), SUM(orcamento_usd = 0),
              SUM(nota_imdb IS NULL), SUM(popularidade IS NULL)
       FROM fact_movies_performance""",
    "SELECT SUM(receita_usd IS NULL AND lucro_usd <> 0) FROM fact_movies_performance",
    "SELECT MIN(nota_imdb), MAX(nota_imdb), MIN(nota_tmdb), MAX(nota_tmdb) FROM fact_movies_performance",
    """SELECT MIN(nota_media_usuarios), MAX(nota_media_usuarios),
              MIN(qtd_avaliacoes_usuarios), MAX(qtd_avaliacoes_usuarios) FROM dim_reviews""",
    "SELECT COUNT(*), COUNT(DISTINCT sk_movie_id) FROM dim_reviews",
    "SELECT MIN(rating), MAX(rating), COUNT(DISTINCT sk_movie_id) FROM movie_reviews",
]

for sql in checagens:
    consultar(sql)

['tipo_pessoa', 'COUNT(*)']
   ('Ator', 273400)
   ('Diretor', 65200)
   ('Roteirista', 86056)

['status_filme', 'COUNT(*)']
   ('Em Produção', 598)
   ('Lançado', 94304)
   ('Planejado', 50)
   ('Pós-Produção', 693)

['MIN(ano_lancamento)', 'MAX(ano_lancamento)']
   (2016, 2029)

['SUM(receita_usd IS NULL)', 'SUM(receita_usd = 0)', 'SUM(orcamento_usd IS NULL)', 'SUM(orcamento_usd = 0)', 'SUM(nota_imdb IS NULL)', 'SUM(popularidade IS NULL)']
   (92272, 0, 87719, 0, 12674, 3615)

['SUM(receita_usd IS NULL AND lucro_usd <> 0)']
   (6296,)

['MIN(nota_imdb)', 'MAX(nota_imdb)', 'MIN(nota_tmdb)', 'MAX(nota_tmdb)']
   (0.0, 10.0, 0.0, 10.0)

['MIN(nota_media_usuarios)', 'MAX(nota_media_usuarios)', 'MIN(qtd_avaliacoes_usuarios)', 'MAX(qtd_avaliacoes_usuarios)']
   (0.0, 10.0, 1, 13)

['COUNT(*)', 'COUNT(DISTINCT sk_movie_id)']
   (40267, 40267)

['MIN(rating)', 'MAX(rating)', 'COUNT(DISTINCT sk_movie_id)']
   (0.0, 10.0, 40267)



## 3. SQLs de referência (gabarito manual)
5 consultas que viraram exemplo do prompt e prova das regras: lucro médio só com receita informada; margem com receita+orçamento; dupla ator-diretor; últimos 5 anos com `date('now')`; mais avaliados com desempate. Repare nas limitações: títulos repetidos (ex: Die Hart), orçamentos de centavos no topo da margem, luta livre no elenco.

In [3]:
sqls_referencia = {
  "1_lucro_medio_por_genero": """
    SELECT g.nome_genero,
           ROUND(AVG(f.lucro_brl), 2) AS lucro_medio_brl,
           COUNT(*) AS qtd_filmes
    FROM fact_movies_performance f
    JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
    JOIN dim_genres g ON g.sk_genre_id = bg.sk_genre_id
    WHERE f.receita_brl IS NOT NULL
    GROUP BY g.nome_genero
    ORDER BY lucro_medio_brl DESC
  """,
  "2_maior_margem_de_lucro": """
    SELECT m.titulo, f.receita_brl, f.orcamento_brl, f.lucro_brl,
           ROUND(f.lucro_brl * 1.0 / f.receita_brl, 4) AS margem
    FROM fact_movies_performance f
    JOIN dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.receita_brl IS NOT NULL
      AND f.orcamento_brl IS NOT NULL
      AND f.receita_brl > 0
    ORDER BY margem DESC
    LIMIT 10
  """,
  "3_dupla_ator_diretor": """
    SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor, COUNT(*) AS qtd_filmes
    FROM bridge_movie_person bpa
    JOIN dim_people a ON a.sk_person_id = bpa.sk_person_id AND a.tipo_pessoa = 'Ator'
    JOIN bridge_movie_person bpd ON bpd.sk_movie_id = bpa.sk_movie_id
    JOIN dim_people d ON d.sk_person_id = bpd.sk_person_id AND d.tipo_pessoa = 'Diretor'
    GROUP BY a.sk_person_id, d.sk_person_id
    ORDER BY qtd_filmes DESC
    LIMIT 5
  """,
  "4_ator_mais_participacoes_ultimos_5_anos": """
    SELECT p.nome_pessoa, COUNT(*) AS participacoes
    FROM dim_movies m
    JOIN bridge_movie_person bp ON bp.sk_movie_id = m.sk_movie_id
    JOIN dim_people p ON p.sk_person_id = bp.sk_person_id AND p.tipo_pessoa = 'Ator'
    WHERE m.data_lancamento >= date('now', 'start of year', '-4 years')
      AND m.data_lancamento <= date('now')
    GROUP BY p.sk_person_id
    ORDER BY participacoes DESC
    LIMIT 5
  """,
  "5_filmes_mais_avaliados_pelos_usuarios": """
    SELECT m.titulo, r.qtd_avaliacoes_usuarios, r.nota_media_usuarios
    FROM dim_reviews r
    JOIN dim_movies m ON m.sk_movie_id = r.sk_movie_id
    ORDER BY r.qtd_avaliacoes_usuarios DESC,
             r.nota_media_usuarios DESC,
             m.titulo
    LIMIT 10
  """,
}

for nome, sql in sqls_referencia.items():
    print("###", nome)
    consultar(sql)

### 1_lucro_medio_por_genero
['nome_genero', 'lucro_medio_brl', 'qtd_filmes']
   ('Science Fiction', 520783718.33, 227)
   ('Adventure', 514678791.39, 393)
   ('Action', 343223021.87, 594)
   ('Fantasy', 335448446.35, 255)
   ('Family', 324648758.33, 264)
   ('Animation', 303517897.37, 239)
   ('War', 194016703.28, 91)
   ('History', 146826836.61, 171)
   ('Comedy', 143589472.55, 813)
   ('Mystery', 119462433.49, 246)
   ('Music', 111985444.28, 100)
   ('Romance', 98645875.99, 312)
   ('Crime', 93988989.27, 294)
   ('Thriller', 92872345.47, 623)
   ('Horror', 92805277.2, 383)
   ('Drama', 84161852.72, 1297)
   ('Tv Movie', 9228474.48, 10)
   ('Documentary', 5028959.44, 153)
   ('Western', -3581373.25, 20)

### 2_maior_margem_de_lucro
['titulo', 'receita_brl', 'orcamento_brl', 'lucro_brl', 'margem']
   ("Dad, I'm Sorry", 95303762.5, 712.12, 95303050.39, 1.0)
   ('Etlb', 3225800, 161.29, 3225638.71, 1.0)
   ('Jailbait', 23778840.8, 1688.44, 23777152.36, 0.9999)
   ('Trivikrama', 52328, 2

## 4. config.py
`MODELOS` em ordem de prioridade, `LIMITE_LINHAS=50` e `TIMEOUT_CONSULTA_S=20` (protegem contexto e travam consulta lenta). `carregar_ambiente()` lê o `.env` e pede a chave com `getpass` se faltar.

In [4]:
from cinedata.config import MODELOS, LIMITE_LINHAS, TIMEOUT_CONSULTA_S
print(MODELOS)
print('limite linhas:', LIMITE_LINHAS, '| timeout:', TIMEOUT_CONSULTA_S)

['nvidia/nemotron-3.5-lightning:free', 'google/gemma-4-26b-a4b-it:free']
limite linhas: 50 | timeout: 20


## 5. banco.py (guardrails, camada 3)
Conexão `mode=ro` + `set_authorizer`: só `SELECT` nas 10 tabelas. `alembic_version` e `sqlite_*` negadas. O erro abaixo é o esperado. CTEs liberadas; `executar_consulta` devolve `{colunas, linhas, truncado}`.

In [5]:
from cinedata.banco import executar_consulta
print(executar_consulta('SELECT COUNT(*) FROM dim_movies'))
try:
    executar_consulta('SELECT * FROM alembic_version')
except Exception as e:
    print('erro esperado:', e)

{'colunas': ['COUNT(*)'], 'linhas': [[95645]], 'truncado': False}
erro esperado: access to alembic_version.version_num is prohibited


## 6. guardrails.py (camada 2)
`validar_sql()` só aceita começar com `SELECT`/`WITH` (tira comentários antes). Não caça DELETE no meio do texto (evita falso alarme em título de filme). Escrita escondida cai no banco ro.

In [6]:
from cinedata.guardrails import validar_sql
exemplos = [
    'SELECT * FROM dim_movies',
    'DELETE FROM dim_movies',
    'WITH x AS (SELECT 1) SELECT * FROM x',
    '/* bloco */ SELECT 1',
]
for sql in exemplos:
    print(repr(sql[:40]), '->', validar_sql(sql))


'SELECT * FROM dim_movies' -> None
'DELETE FROM dim_movies' -> Só consultas de leitura são permitidas: o SQL deve começar com SELECT (ou WITH ... SELECT). Não é possível inserir, alterar ou apagar dados.
'WITH x AS (SELECT 1) SELECT * FROM x' -> None
'/* bloco */ SELECT 1' -> None


## 7. modelos.py (fallback)
Só troca de modelo com 429 do **provider lotado**. Cota da conta (`free-models-per-day/min`) e 401 não trocam; SQL errado é retry. `max_retries=0` (1 falha = 1 request). Teste real de tool calling em 05/10 (`python src/cinedata/tests/teste_tool_calling.py`, ~2 requests/modelo): Nemotron OK, `glm-5.2:free` 404 removido, Gemma 429 mantido como fallback. Testes locais sem cota: `python -m pytest` (28 passed).

In [7]:
from cinedata.modelos import provider_lotado, limite_da_conta
from pydantic_ai.exceptions import ModelHTTPError
prov = ModelHTTPError(429, 'm', {'error': {'message': 'Provider returned error', 'code': 429}})
cota = ModelHTTPError(429, 'm', {'error': {'message': 'Rate limit exceeded: free-models-per-day', 'code': 429}})
print('provider lotado -> fallback:', provider_lotado(prov))
print('cota da conta -> fallback:', provider_lotado(cota))

provider lotado -> fallback: True
cota da conta -> fallback: False


## 8. prompt.py (guardrails, camada 1)
`PROMPT_SISTEMA` com esquema fixo das 10 tabelas (avisar na apresentação), 2 exemplos enxutos, regras: R$ padrão, margem=lucro/receita + aviso <R$10mil, últimos 5 anos até hoje, IMDb padrão, desempate, ano sempre, aviso de duplicadas/base pequena. `montar_prompt()` injeta a data real.

In [8]:
from cinedata.prompt import montar_prompt
p = montar_prompt("2026-10-05")
print("data ok:" , "2026-10-05" in p)
print("sem placeholder:", "__DATA_HOJE__" not in p)
print(p[:300])

data ok: True
sem placeholder: True
Você é o analista de dados do CineData Analytics. Responda em português, consultando
o banco SQLite com a ferramenta de SQL.

ESCOPO E SEGURANÇA
- Responda só sobre o catálogo de filmes (filmes, elenco e equipe, gêneros, produtoras,
  receita, notas, avaliações). Para outro assunto, responda apenas:


## 9. cache.py
Normalização (minúsculas, 1 espaço, sem acentos, sem `?!.`), JSON `{pergunta: resposta}` em `data/cache.json`, `/limpar` apaga. Só sem histórico; recusa e erro nunca entram. Teste com arquivo temporário (0 requests).

In [9]:
from cinedata.cache import normalizar_pergunta, guardar_cache, consultar_cache, limpar_cache
from pathlib import Path
tmp = Path("cache_teste.json")
if tmp.exists(): tmp.unlink()
print(normalizar_pergunta("  Qual o FILME mais LONGO?? "))
guardar_cache("Qual o filme mais longo?", "Teste", caminho=tmp)
print(consultar_cache("qual o filme mais longo", caminho=tmp))
print(consultar_cache("outra pergunta", caminho=tmp))
limpar_cache(caminho=tmp)
print("limpo:", not tmp.exists())

qual o filme mais longo
Teste
None
limpo: True


## 10. agente.py + trace.py
`criar_agente()`: `Agent(FallbackModel, instructions, retries=2)` com 1 tool `consultar_banco(sql)`. Erro vira texto p/ corrigir (retry, não fallback). `perguntar` (terminal, sync) e `perguntar_async` (notebook, await). `mostrar_trace` imprime o ciclo ReAct + uso.

In [10]:
from cinedata.agente import criar_agente
a = criar_agente()
print("agente:", type(a).__name__)
nomes = []
for ts in getattr(a, "toolsets", []):
    nomes += list(getattr(ts, "tools", {}).keys())
print("tools:", nomes if nomes else "não listado nesta versão (ok — o teste real é o perguntar)")

agente: Agent
tools: ['consultar_banco']


## 11. Demo (gasta ~2-3 requests)
No notebook usar sempre `await perguntar_async` (`run_sync` dá `RuntimeError` no Jupyter). Repetindo a mesma pergunta sem histórico volta do cache (0 requests).

In [11]:
import importlib, cinedata.agente
importlib.reload(cinedata.agente)
from cinedata.agente import perguntar_async
from cinedata.trace import mostrar_trace
resp, res = await perguntar_async("Quantos filmes há no catálogo?")
print(resp)
if res: mostrar_trace(res)

                 pydantic-ai v2.54.0 • Python 3.13.3

      / \        agent: agente
     /   \         model: fallback:openrouter:nvidia/nemotron-…router:google/gemma-4-26b-a4b-it:free
   /___.___\       tools: 1 • capabilities: 0
  /    |    \
/      |      \  observability: off — see every model and tool call live, with cost
`---.._|_..---'    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
                   or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


No catálogo do CineData Analytics, existem **95.645 filmes** registrados na tabela `dim_movies`.

**Detalhes importantes:**
- **Títulos únicos:** 88.038 (o catálogo contém **7.607 entradas duplicadas**, conforme a regra de aviso sobre títulos repetidos).
- **Dados financeiros:** Apenas ~3.373 filmes têm receita informada e ~1.630 têm tanto receita quanto orçamento disponíveis.
- **Moeda:** As contas financeiras (receita, orçamento, lucro) estão em **R$ (reais)** por padrão, conforme o esquema das colunas `_brl`.

Critérios usados: contagem total de linhas em `dim_movies`, com verificação de títulos distintos para alertar sobre duplicidades, conforme as diretrizes do catálogo.
[USUÁRIO]      Quantos filmes há no catálogo?
[MODELO→TOOL]  consultar_banco({'sql': 'SELECT COUNT(*) AS total_filmes FROM dim_movies;'})
[TOOL→MODELO]  consultar_banco retornou: colunas: ['total_filmes']
linhas (1): [[95645]]
[MODELO→TOOL]  consultar_banco({'sql': 'SELECT COUNT(DISTINCT titulo) AS titulos_unicos 

## 12. cli.py
Terminal `cinedata` (estilo opencode, só stdlib): cores ANSI + spinner, `/ajuda /limpar /sair`, janela de 4 trocas, hit mostra `(do cache)`. Não executado aqui — rode no terminal: `cinedata`. Sem instalar: `python -m cinedata.cli`.
## Entrega
- Código: `src/cinedata/` + testes (`python -m pytest`, 28 passed)
- Banco fora do repo: baixar `cinerocket.db` para `src/cinedata/data/`
- Limitações na apresentação: duplicadas, orçamentos baixos, lucro NULL→zero, base pequena, luta livre; extras fora: gráficos, busca semântica, avaliação automática, Databricks.